In [ ]:
################################ Category mapping and type tag mapping function ##################################
import json
import csv
import re
import time
import openai
import requests
import pandas as pd
from bs4 import BeautifulSoup
# from googlenewsdecoder import GoogleDecoder
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
from nltk import pos_tag, word_tokenize
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import requests
import urllib.parse
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

project_id = 124

LABOUR_NEWS_TAG = ["Labour Strikes", "Strikes", "Quits", "unrest", "agitation", "lock out", "termination", "ESI",
                   "unrest", "Fired", "Quit", "suicide", "protest", "injury", "injured", "salary", "EPF", "attrition",
                   "Dispute", "harassment", "protests", "Strike"]

RAID_NEWS_TAG = ["Raid", "Raids", "Raided"]

RESIGNATION_NEWS_TAG = ["Resign", "Resigns", "Resigned", "Resignation", "resigning", "management changes", "terminate"]

FRAUD_REGULATORY_ACTION_NEWS_TAG = ["Fraud", "Embezzlement", "Money laundering", "Syphoning", "CBI inquiry", "Bribery",
                                    "Political Influence", "SEBI proceeding", "regulatory action", "Penalties", "Fines",
                                    "Penalty", "Cheating", "NPA", "Default", "Criminal", "Breach", "Scam", "Arrest",
                                    "Warrant Issued", "Probe", "Fined", "Misappropriation", "Inspection",
                                    "Complaint against", "insolvency", "I T notice", "Hawala", "CBI", "FIR", "Jail",
                                    "Illeagal", "Forging", "booked", "scrutinize", "land grab", "RBI", "CBI", "SEBI",
                                    "IRDA", "EXIM", "Fraudulent", "theft", "theif", "bankruptcy", "Black Money",
                                    "Bribe",
                                    "Central Bureau of Investigation", "charge sheet", "chargesheet", "cheat",
                                    "corrupt",
                                    "corruption", "custody", "Due", "Dues", "evading", "Fine", "Fined", "forgery",
                                    "illegal", "Inspect", "Inspector", "investigation", "Labour Dispute", "land grab",
                                    "non compliance", "notices", "Offered Money", "Penalized", "police",
                                    "Serious Fraud",
                                    "SFIO", "violation", "Whistleblower"]

MARKET_STOCK_MOVEMENT_NEWS_TAG = ["plunges", "Slowdown", "Delay", "Delayed", "downgrade", "downgraded", "losses",
                                  "loss", "SEBI", "profit decline", "Wind Up", "plummets", "Decline", "exits", "low",
                                  "pledge", "plummet", "SELL Recommendation", "stake sale", "stress", "suspends",
                                  "worst", "sell", "fall", "down", "drop", "falls", "sells"]

LITIGATION_NEWS_TAG = ["Case Registered", "Court Denies", "Court Rejects", "contempt", "Registered Case",
                       "Settlement", "allegations", "High Court", "District Court", "Lower Court", "Supreme Court",
                       "Judgement", "copyrights", "defamation", "DRT", "NCLT"]

WHISTLEBLOWER_NEWS_TAG = ["Whistleblower"]

ACCIDENT_NEWS_TAG = ["Accident", "Fire", "Death", "Died", "Passed Away", "Die", "casualty", "mishap", "Blast",
                     "casualties", "Died", "killed", "kills", "passes away", "shut down"]

# # Load the sentiment words from CSV
# csv_file_path = r'D:\Ronit_Projects\DailyNews\DailyNews_4\Related_files\SentimentalWords.csv'
# df_csv = pd.read_csv(csv_file_path)
# sentiment_mapping = {'Negative': -1, 'Neutral': 0, 'Positive': 1}
# word_sentiments_csv = dict(zip(df_csv['text'], df_csv['sentiment'].map(sentiment_mapping)))
# sia = SentimentIntensityAnalyzer()
#
#
# # Function to get sentence sentiment
# def get_sentence_sentiment(sentence):
#     words = word_tokenize(sentence.lower())
#     csv_sentiment = sum(word_sentiments_csv.get(word, 0) for word in words)
#     # print("csv_sentiment : ", csv_sentiment)
#     nlp_sentiment = sia.polarity_scores(sentence)['compound']
#     # print("nlp_sentiment : ", nlp_sentiment)
#     combined_sentiment = csv_sentiment + nlp_sentiment
#
#     if combined_sentiment > 0:
#         return 'Positive'
#     elif combined_sentiment < 0:
#         return 'Negative'
#     else:
#         return 'Neutral'








def call_openai_model(prompt, model, retries, timeout):
    for attempt in range(1, retries + 1):
        try:
            response = openai.ChatCompletion.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                timeout=timeout
            )

            content = response['choices'][0]['message']['content'].strip()

            # Remove markdown-wrapped JSON if present
            if content.startswith("```"):
                content = content.split("```")[1].strip()
                if content.startswith("json"):
                    content = content[len("json"):].strip()

            result = json.loads(content)
            return result["sentiment"]

        except json.JSONDecodeError:
            raise ValueError(f"⚠️ Invalid JSON returned by {model}:\n{content}")

        except (openai.error.Timeout,
                openai.error.APIConnectionError,
                openai.error.RateLimitError,
                openai.error.ServiceUnavailableError,
                openai.error.OpenAIError,
                OSError) as e:

            print(f"⏳ [{model}] Attempt {attempt}/{retries} failed: {e}")
            if attempt < retries:
                time.sleep(5)
            else:
                return None  # Allow fallback

        except Exception as e:
            raise RuntimeError(f"❌ Unexpected error with {model}: {e}")

    return None


def get_sentence_sentiment(sentence, retries=5, timeout=60):
    prompt = f"""
    Classify the following news sentence for sentiment.

    Choose strictly one of: Positive, Negative, or Neutral.

    Respond strictly in this JSON format:
    {{
      "sentiment": "Your Sentiment"
    }}

    Sentence: "{sentence}"
    Respond only with the JSON. No explanation.
    """

    # Primary: gpt-3.5-turbo
    sentiment = call_openai_model(prompt, "gpt-3.5-turbo", retries, timeout)

    # Fallback: gpt-4o
    if sentiment is None:
        print("🔁 Falling back to gpt-4o...")
        sentiment = call_openai_model(prompt, "gpt-4o", retries, timeout)

    if sentiment is None:
        raise TimeoutError("❌ Request failed on both gpt-3.5-turbo and gpt-4o after multiple retries.")

    return sentiment


def get_category_tag(sentence):
    define_tags = (LABOUR_NEWS_TAG + RAID_NEWS_TAG + RESIGNATION_NEWS_TAG + FRAUD_REGULATORY_ACTION_NEWS_TAG +
                   MARKET_STOCK_MOVEMENT_NEWS_TAG + LITIGATION_NEWS_TAG + WHISTLEBLOWER_NEWS_TAG + ACCIDENT_NEWS_TAG)

    tokens = word_tokenize(sentence)
    tagged_words = pos_tag(tokens)

    verbs = [word for word, pos in tagged_words if pos.startswith('V')]
    adjectives = [word for word, pos in tagged_words if pos.startswith('J')]
    adverbs = [word for word, pos in tagged_words if pos.startswith('R')]
    nouns = [word for word, pos in tagged_words if pos.startswith('N')]
    all_tags = verbs + adjectives + adverbs + nouns
    filtered_tags = [tag.title() for tag in all_tags if tag.title() in define_tags or tag in define_tags]

    if filtered_tags:
        return filtered_tags[0]
    return 'Other'


def get_type_tag(category):
    with open(r'D:\Nexensus_Projects\News\news_type_tag.csv', mode='r') as file:
        reader = csv.DictReader(file)
        for row in reader:
            if row['Category'] == category:
                return row['type_tag'].title()
        return "Other"

In [ ]:
###################### Narcotics india news scraping code(Not for integeration) ##################################################################
import requests
from bs4 import BeautifulSoup
import json

newsSite = "Narcotics"
def write_to_txt_resolve(newData):
    file1 = r'D:\Nexensus_Projects\News\\data_{}.txt'.format(newsSite)
    text_file = open(file1, 'a')
    text_file.write(json.dumps(newData))
    text_file.write(",")
    text_file.write("\n")
    text_file.close()


header = {
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/112.0.0.0 Safari/537.36",
        "Accept" : "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
        "Cookie": '_ht_fp=WINHTNEW53801749811666; ppid=4192611473d3d243d10177b00806db7e5b5245baa725924aebdab8af2840be1a; _cb=sQzDtCaysICBMmSyz; _lc2_fpi=3423e49eb244--01jxmee7ttpcsc8wmn5qqdd9qt; _lc2_fpi_meta=%7B%22w%22%3A1749811666779%7D; _gcl_au=1.1.1027468645.1749811667; usercohortcitynew=Delhi; _ga=GA1.1.1226154917.1749811667; _cc_id=392a0ad35f69aeaa5a086268534a6b42; _scor_uid=70d644981d24478da4837753ae8a84da; _vwo_uuid_v2=D04DB616A21F3BC47D77E867A831DD341|d71b91edf2fc2e981fd978fba8cd07a1; _vwo_uuid=D04DB616A21F3BC47D77E867A831DD341; _vis_opt_s=1%7C; _clck=yaa34%7C2%7Cfy9%7C0%7C2045; FirstTimeVisitCookie=here; pubmatic-unifiedid=%7B%22TDID%22%3A%221f59fed1-dfce-46ab-85f6-0dd51badad26%22%2C%22TDID_LOOKUP%22%3A%22FALSE%22%2C%22TDID_CREATED_AT%22%3A%222025-08-12T11%3A40%3A32%22%7D; pubmatic-unifiedid_cst=zix7LPQsHA%3D%3D; ht-location=IN; Meta-Geo=IN--DL--NEWDELHI; HTMyOffer=myOfferHT; prContextualTags=BikeKeywords; _li_dcdm_c=.hindustantimes.com; panoramaId_expiry=1756807605986; _sp_ses.e8bf=*; cdpecho_country=IN; cdpecho_city=New Delhi; oneTap_viewed=true; g_oneTapShow=true; moe_uuid=c665ece9-08a0-44da-995e-d6b050ca68a5; gptScript=true; cdpCountrytrack=true; _chartbeat2=.1749811666504.1756722185974.0000000000000001.D36Tp9CmuNHICzXZcqW67ygT6JCu.1; _cb_svref=external; __gads=ID=d92bedca347cfd36:T=1749811667:RT=1756722185:S=ALNI_Ma0Bxtorm-9r3HmLPZNNqWEixp9tg; __gpi=UID=0000112ce9bfde98:T=1749811667:RT=1756722185:S=ALNI_MbQT5Y7BVauwWjOrlJTedqnrual7A; __eoi=ID=382fbc027fc564db:T=1749811667:RT=1756722185:S=AA-AfjYvQfj6L6Kn5Xa7nKZzDiuh; _sp_id.e8bf=ad71a14e-2ff0-4105-84ff-dd5201c2ce01.1749811667.30.1756722187.1755227920.974ea0ba-3d7b-4cc4-86bd-ed3d4a999375; _ga_9CYSK4EP0B=GS2.1.s1756721207$o31$g1$t1756722188$j60$l0$h0; moe_c_s=1; moe_u_d=%7B%22a%22%3A%5B%5D%2C%22s_old%22%3Afalse%2C%22d_id%22%3A%22c665ece9-08a0-44da-995e-d6b050ca68a5%22%2C%22d_added%22%3Atrue%7D; moe_s_n=%7B%22s_k%22%3A%2255c93a32-91bc-439a-92d9-6830d3b58a39%22%2C%22s_s_t%22%3A%222025-09-01T10%3A07%3A00.051Z%22%2C%22s_m_t%22%3A1800%2C%22c_i_t_t%22%3A%5B%5D%2C%22s_e_t%22%3A1756723997765%2C%22n_o_s%22%3A29%7D; cdp_page_referrer=; idleSliderShown=true',
        }

soup = BeautifulSoup(requests.get("https://narcoticsindia.nic.in/news.php").content, 'html.parser')
table_body = soup.find("table").find("tbody")
for row in table_body.find_all('tr'):
    title = row.find_all('td')[1].text.strip()
    link = row.find_all('td')[2].find('a')['href']
    print(requests.get(link, headers=header))
    newsData = {"heading": title, "link" : link, }
    print(newsData)
    write_to_txt_resolve(newsData)
    print("-------------------------------------------------------")
    

<Response [200]>
{'heading': 'NCB And RRU Sign MoU to Boost Research and Training on Drug Crimes', 'link': 'https://www.newsonair.gov.in/ncb-and-rru-sign-mou-to-boost-research-and-training-on-drug-crimes/'}
-------------------------------------------------------
<Response [200]>
{'heading': 'NCB, SSB to Strengthen Anti-Drug Trafficking Operations on Indo-Nepal Border', 'link': 'https://www.newsonair.gov.in/ncb-ssb-to-strengthen-anti-drug-trafficking-operations-on-indo-nepal-border/'}
-------------------------------------------------------
<Response [200]>
{'heading': 'Two doctors held for supplying tramadol tablets to private hospitals in Amritsar', 'link': 'https://www.tribuneindia.com/news/amritsar/two-doctors-held-for-supplying-tramadol-tablets-to-private-hospitals-in-amritsar/'}
-------------------------------------------------------
<Response [200]>
{'heading': 'Jharkhand: NCB freezes Rs 48.6 lakh assets of drug traffickers in poppy straw smuggling case', 'link': 'https://www.anin

KeyboardInterrupt: 

In [ ]:
################################## SEBI news scraping code ################################################################
import requests
from bs4 import BeautifulSoup
import time
import json
import datetime
import os

newsSite = "SEBI"
endScraping = False
# def write_to_txt_resolve(newData):
#     file1 = r'D:\Nexensus_Projects\News\\data_{}.txt'.format(newsSite)
#     text_file = open(file1, 'a')
#     text_file.write(json.dumps(newData))
#     text_file.write(",")
#     text_file.write("\n")
#     text_file.close()

# === Load existing JSON lines to avoid duplicates ===
file_path = r'D:\Nexensus_Projects\News\\data_{}.txt'.format(newsSite)
existing_news = set()
if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip().rstrip(",")
            if not line:
                continue
            try:
                # Normalize JSON to consistent string (sorted keys, no spaces)
                normalized = json.dumps(json.loads(line), sort_keys=True, separators=(",", ":"))
                existing_news.add(normalized)
            except json.JSONDecodeError:
                continue

# === Safe write function ===
def write_to_txt_resolve(newData):
    normalized = json.dumps(newData, sort_keys=True, separators=(",", ":"))
    if normalized in existing_news:
        print("⏩ Skipping duplicate news item.")
        return

    existing_news.add(normalized)
    with open(file_path, 'a', encoding='utf-8') as text_file:
        text_file.write(json.dumps(newData))
        text_file.write(",\n")


url = "https://www.sebi.gov.in/sebiweb/ajax/home/getnewslistinfo.jsp"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.sebi.gov.in/sebiweb/home/HomeAction.do?doListing=yes&sid=6&ssid=23&smid=0"
}

cookies = {
    "JSESSIONID": "C7F7C8E8722D1736110467AA0D643BD3"
}

# cookies = {
#   "_ga": "GA1.3.1771182332.1755684907",
#   "_gid": "GA1.3.1846028703.1757481148",
#   "_ga_9HZ5J5Q3K5": "GS2.3.s1757486376$o13$g0$t1757486376$j60$l0$h0"
# }


# Base payload (except nextValue which we will change)
base_payload = {
    "next": "n",
    "search": "",
    "fromDate": "",
    "toDate": "",
    "fromYear": "",
    "toYear": "",
    "deptId": "-1",
    "sid": "6",
    "ssid": "23",
    "smid": "0",
    "ssidhidden": "23",
    "intmid": "-1",
    "sText": "Media & Notifications",
    "ssText": "Press Releases",
    "smText": "",
    # "doDirect": "3"
}

# Loop through pages
for page in range(0, 250):  # scrape first 231 pages
    if endScraping == True:
        print("So ending scraping data")
        break
    time.sleep(1)
    payload = base_payload.copy()
    payload["nextValue"] = str(page)
    payload['doDirect'] = str(page)

    r = requests.post(url, headers=headers, cookies=cookies, data=payload)
    print(f"Page {page} -> {len(r.text)} characters")
    # print(r.content)
    soup = BeautifulSoup(r.content, 'html.parser')
    # print(soup.find('table', {'id' : 'sample_1'}).find_all('tr')[0])
    try:
        for row in soup.find('table', {'id' : 'sample_1'}).find_all('tr')[1:]:
            date = row.find_all('td')[0].text
            date = datetime.datetime.strptime(date, '%b %d, %Y')  # parse string to datetime

            # ✅ Check if date is within the last 5 years from today
            today = datetime.datetime.now()
            five_years_ago = datetime.datetime(today.year - 5, 1, 1)
            # five_years_ago = today.replace(year=today.year - 5)

            if date >= five_years_ago:
                date = date.strftime('%Y-%m-%d')
                print("✅ Date within last 5 years:", date)
            else:
                endScraping = True
                print(f"❌ Date {date.strftime('%Y-%m-%d')} is older than 5 years")
                break
                # print(f"❌ Date {date.strftime('%Y-%m-%d')} is older than 5 years")
            # date = date.strftime('%Y-%m-%d')  # format to desired output
            pr_no = row.find('th').text
            # title  = row.find_all('td')[1].text.replace("\u2018", "").replace("\u2019", "").replace("\u2013", "").replace(
            #                     "\u00a3", "").replace("\u00a0", "").strip()
            title = row.find_all('td')[1].text.strip()
            link = row.find_all('td')[1].find('a')['href']
            category = get_category_tag(title)
            typeTag = get_type_tag(title)
            print('date  :', date)
            print('pr_no :', pr_no)
            print('title :', title)
            print('href  :', link)
            newsData = {"companyName": None, "cin": None, "heading": title, "newsDate": date, "newsBody": None,  "link" : link, "category": category, "typeTag" : typeTag, "sentiment": None}
            # newsData = {"companyName": None, "cin": None, "heading": title, "newsDate": date, "newsBody": None,  "link" : link, "pr_no": pr_no, "category": category, "typeTag" : typeTag, "sentiment": None}
            print(newsData)
            write_to_txt_resolve(newsData)
    except Exception as e:
        print(e)
        pass
    # parse r.text with BeautifulSoup or regex if it's HTML


Page 0 -> 13533 characters
✅ Date within last 5 years: 2025-10-31
date  : 2025-10-31
pr_no : 67/2025
title : Niveshak Shivir to be held on November 01, 2025 in the city of Amritsar, Punjab
href  : https://www.sebi.gov.in/media-and-notifications/press-releases/oct-2025/niveshak-shivir-to-be-held-on-november-01-2025-in-the-city-of-amritsar-punjab_97576.html
{'companyName': None, 'cin': None, 'heading': 'Niveshak Shivir to be held on November 01, 2025 in the city of Amritsar, Punjab', 'newsDate': '2025-10-31', 'newsBody': None, 'link': 'https://www.sebi.gov.in/media-and-notifications/press-releases/oct-2025/niveshak-shivir-to-be-held-on-november-01-2025-in-the-city-of-amritsar-punjab_97576.html', 'category': 'Other', 'typeTag': 'Other', 'sentiment': None}
⏩ Skipping duplicate news item.
✅ Date within last 5 years: 2025-10-10
date  : 2025-10-10
pr_no : 66/2025
title : Ease of doing business  Rationalisation and standardisation of penalties  levied on stock brokers by stock exchanges
href  

In [4]:
##################################### CBI News scraping Code ##############################################################
import requests 
from bs4 import BeautifulSoup
import time
import json
import datetime
import os
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

newsSite = "CBI"

# def write_to_txt_resolve(newData):
#     file1 = r'D:\Nexensus_Projects\News\\data_{}.txt'.format(newsSite)
#     text_file = open(file1, 'a')
#     text_file.write(newData)
#     text_file.write(",")
#     text_file.write("\n")
#     text_file.close()

# === Load existing JSON lines to avoid duplicates ===
file_path = r'D:\Nexensus_Projects\News\\data_{}.txt'.format(newsSite)
existing_news = set()
if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip().rstrip(",")
            if not line:
                continue
            try:
                # Normalize JSON to consistent string (sorted keys, no spaces)
                normalized = json.dumps(json.loads(line), sort_keys=True, separators=(",", ":"))
                existing_news.add(normalized)
            except json.JSONDecodeError:
                continue

# === Safe write function ===
def write_to_txt_resolve(newData):
    normalized = json.dumps(newData, sort_keys=True, separators=(",", ":"))
    if normalized in existing_news:
        print("⏩ Skipping duplicate news item.")
        return

    existing_news.add(normalized)
    with open(file_path, 'a', encoding='utf-8') as text_file:
        text_file.write(json.dumps(newData))
        text_file.write(",\n")


response = requests.get("https://cbi.gov.in/press-releases")
soup = BeautifulSoup(response.content, 'html.parser')

table_body = soup.find('table', {'id' : 'commonDataTable'}).find('tbody')

thirty_days_ago = datetime.datetime.now() - datetime.timedelta(days=30)
for row in table_body.find_all('tr'):
    # print(row)
    title = row.find_all('td')[1].text.replace("News Heading:", "").replace("\u2018", "").replace("\u2019", "").replace("\u2013", "").replace(
                            "\u00a3", "").replace("\u00a0", "").strip()
    date = row.find_all('td')[2].text.replace("News Date:", "").strip()
    link = row.find_all('td')[1].find("a")['href']
    date = datetime.datetime.strptime(date, '%d/%m/%Y')
    date = date.strftime('%Y-%m-%d')
    print("DD/MM/YYYY →", date)
    if datetime.datetime.strptime(date, "%Y-%m-%d") < thirty_days_ago:
        break
    # print("title :", title)
    # print("date :", date)
    # print("link :", link)
    try:
        # Create a session to persist cookies
        session = requests.Session()
        # Step 1: Set language to English
        lang_url = "https://cbi.gov.in/lang/en"
        session.get(lang_url, headers=headers, timeout=10)

        # Step 2: Now request the target page (will come in English)
        des_res = session.get(link, headers=headers, timeout=20)
        # des_res = requests.get(link,headers=headers, timeout=20)
        des_soup =  BeautifulSoup(des_res.content, 'html.parser')
        description = des_soup.find('div', 'press-txt').text.replace("\u2018", "").replace("\u2019", "").replace("\u2013", "").replace(
                            "\u00a3", "").replace("\u00a0", "").replace("\u021d", "").replace("\u2014", "").replace("\u200b", "").replace("\u201c", "").replace(
                                "\u20b9", "").replace("\u2022", "").replace("\u1e62", "").replace("\u201d","").strip()
        # print("description :", description)
    except Exception as e:
        print('e :', e)
        time.sleep(20)
        # Create a session to persist cookies
        session = requests.Session()
        # Step 1: Set language to English
        lang_url = "https://cbi.gov.in/lang/en"
        session.get(lang_url, headers=headers, timeout=10)

        # Step 2: Now request the target page (will come in English)
        des_res = session.get(link, headers=headers, timeout=20)
        # des_res = requests.get(link,headers=headers, timeout=20)
        des_soup =  BeautifulSoup(des_res.content, 'html.parser')
        description = des_soup.find('div', 'press-txt').text.replace("\u2018", "").replace("\u2019", "").replace("\u2013", "").replace(
                            "\u00a3", "").replace("\u00a0", "").replace("\u021d", "").replace("\u2014", "").replace("\u200b", "").replace("\u201c", "").replace(
                                "\u20b9", "").replace("\u2022", "").replace("\u1e62", "").replace("\u201d","").strip()
        # print("description :", description)
    # print({"heading": title, "newsDate": date, "link" : link, "newsBody": description})
    category = get_category_tag(title)
    typeTag = get_type_tag(title)
    # newsData = {"heading": title, "newsDate": date, "link" : link, "newsBody": description}
    newsData = {"companyName": None, "cin": None, "heading": title, "newsDate": date, "newsBody": description,  "link" : link, "category": category, "typeTag" : typeTag, "sentiment": None}
    # write_to_txt_resolve(json.dumps(newsData))
    write_to_txt_resolve(newsData)
    time.sleep(2)
    # print("------------")


DD/MM/YYYY → 2025-11-01
DD/MM/YYYY → 2025-11-01
DD/MM/YYYY → 2025-10-31
DD/MM/YYYY → 2025-10-31
DD/MM/YYYY → 2025-10-31
DD/MM/YYYY → 2025-10-25
DD/MM/YYYY → 2025-10-25
DD/MM/YYYY → 2025-10-18
DD/MM/YYYY → 2025-10-18
DD/MM/YYYY → 2025-10-17
DD/MM/YYYY → 2025-10-17
DD/MM/YYYY → 2025-10-17
DD/MM/YYYY → 2025-10-17
DD/MM/YYYY → 2025-10-16
DD/MM/YYYY → 2025-10-16
DD/MM/YYYY → 2025-10-15
DD/MM/YYYY → 2025-10-15
DD/MM/YYYY → 2025-10-15
DD/MM/YYYY → 2025-10-15
DD/MM/YYYY → 2025-10-14
DD/MM/YYYY → 2025-10-14
DD/MM/YYYY → 2025-10-14
DD/MM/YYYY → 2025-10-14
DD/MM/YYYY → 2025-10-13
DD/MM/YYYY → 2025-10-13
DD/MM/YYYY → 2025-10-12
DD/MM/YYYY → 2025-10-10
DD/MM/YYYY → 2025-10-09
DD/MM/YYYY → 2025-10-09
DD/MM/YYYY → 2025-10-08
DD/MM/YYYY → 2025-10-08
DD/MM/YYYY → 2025-10-08
DD/MM/YYYY → 2025-10-07
DD/MM/YYYY → 2025-10-07
DD/MM/YYYY → 2025-10-07
DD/MM/YYYY → 2025-10-04


In [1]:
# Load News model
import spacy
import re
NER = spacy.load("en_core_web_sm")
NER = spacy.load("en_core_web_lg")


COMPANY_KEYWORDS = r"(?:Pvt. Ltd.|Pvt Ltd|Private Limited|Ltd.|Ltd|Limited|LLP|PLC|Bank)"

def extract_companies(text: str):
    """Extract company names based on suffix patterns."""
    companies = set()
    # print(text)
    regex_patterns = [
        # r"(?:M/s\.?\s*)?[A-Z][A-Za-z0-9&.,\- ]+\s+(?:Limited|Ltd\.?|Pvt\.? Ltd\.?|LLP|PLC|Incorporated|Corporation|Company|Co\.?)\b"
        rf"(?:M/s\.?|Messrs)\s+[A-Z][A-Za-z0-9&.\- ]+?{COMPANY_KEYWORDS}",  # M/s ABC Pvt Ltd
        rf"[A-Z][A-Za-z0-9&.\- ]+?{COMPANY_KEYWORDS}"  # ABC Pvt Ltd
        # rf"(?<!M/s\.?\s)(?<!Messrs\s)[A-Z][A-Za-z0-9&.\- ]+?{COMPANY_KEYWORDS}"
    ]
    for pattern in regex_patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        for m in matches:
            # clean_name = m.replace("M/s.", "").replace("M/s", "").replace("Messrs", "").strip(" ,.-")
            if m.startswith("s ") or m.startswith("s."):
                # print("m :", text)
                continue
            clean_name = re.sub(r"^(M/s\.?|Messrs)\s+", "", m.strip(), flags=re.IGNORECASE)
            # print(clean_name)
            companies.add(clean_name)
    # print(companies)
    return list(companies)
# extract_companies("Order in the Matter of Jeevan Dhara Geomine Limited")


def spacy_large_ner(document):
    named_entities_set = {(ent.text.strip(), ent.label_) for ent in NER(document).ents}
    # print(named_entities_set)
    return [entity[0] for entity in named_entities_set if entity[1] == 'ORG']
    # return [entity[0] for entity in named_entities_set if entity[1] in ['ORG','PRODUCT', 'PERSON']]
    # return [entity[0] for entity in named_entities_set if entity[1] == 'ORG' or entity[1] == 'PRODUCT' or entity[1] == 'PERSON']

inp = "A.O., Aditya Birla Housing Finance Ltd"
# inp = "Order in the matter of M/s Weird Infrastructure Corporation limited"
# inp = "Order in the matter of PAFL Industries limited"
for x in spacy_large_ner(inp):
    print(x)

Aditya Birla Housing Finance Ltd


In [3]:

inp = "Madhyachal Vidyut Vitran Nigam Ltd"
# inp = "Order in the matter of M/s Weird Infrastructure Corporation limited"
# inp = "Order in the matter of PAFL Industries limited"
for x in spacy_large_ner(inp):
    print(x)

Nigam Ltd


In [3]:
import re
import json
import pandas as pd

# === Read file ===
with open(r"D:\Nexensus_Projects\News\data_SEBI.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

# === Prepare storage ===
records = []

# === Process each JSON line ===
for line in lines:
    line = line.strip()
    json_str = re.sub(r',\s*$', '', line)  # remove last trailing comma if present
    if not json_str:
        continue

    try:
        data = json.loads(json_str)
        heading = data.get('heading', '')

        # Run NER on heading
        for entity in spacy_large_ner(heading):
            companies = extract_companies(entity)
            if len(companies) >= 1:
                records.append({
                    "heading": heading,
                    # "extracted_companies": ", ".join(companies)
                    "extracted_companies": companies
                })
            else:
                records.append({'heading' : heading, 
                                "extracted_companies" : None})

    except json.JSONDecodeError as e:
        print(f"Skipping invalid JSON line: {e}")
        continue

# === Convert to DataFrame ===
df = pd.DataFrame(records)

# === Save to Excel ===
output_path = r"D:\Nexensus_Projects\News\extracted_companies.xlsx"
df.to_excel(output_path, index=False)

print(f"✅ Data saved to {output_path}")


✅ Data saved to D:\Nexensus_Projects\News\extracted_companies.xlsx


In [ ]:
##################################### NIA News scraping Code(Not for integeration) ##############################################################
import requests
from bs4 import BeautifulSoup
import json

newsSite = "NIA"
def write_to_txt_resolve(newData):
    file1 = r'D:\Nexensus_Projects\News\\data_{}.txt'.format(newsSite)
    text_file = open(file1, 'a')
    text_file.write(json.dumps(newData))
    text_file.write(",")
    text_file.write("\n")
    text_file.close()

url = "https://nia.gov.in/press-releases.htm"

session = requests.Session()
resp = session.get(url)
soup = BeautifulSoup(resp.text, "html.parser")
for row in soup.find('tbody').find_all('tr'):
    s_no = row.find_all('td')[0].text.strip()
    newsDate = row.find_all('td')[1].text.strip()
    unit = row.find_all('td')[2].text.strip()
    category = row.find_all('td')[3].text.strip()
    heading = row.find_all('td')[4].text.strip()
    link = "https://nia.gov.in" +row.find_all('td')[5].find('a')['href']
    # print({"s_no": s_no, "heading": heading, "newsDate": newsDate, "link" : link})
    newsData = {"heading": heading, "newsDate": newsDate, "link" : link}
    print(newsData)
    write_to_txt_resolve(newsData)
    print("-----------------------------------------")

def get_hidden_fields(soup):
    fields = {}
    for field in ["__VIEWSTATE", "__VIEWSTATEGENERATOR", "__EVENTVALIDATION", '__VIEWSTATEENCRYPTED']:
        tag = soup.find("input", {"id": field})
        if tag:
            fields[field] = tag["value"]
    # print(fields.keys())
    return fields

for i in range(25):
    # Extract hidden fields from first page
    hidden = get_hidden_fields(soup)

    # Example: go to page 2
    payload = {
        **hidden,
        "__EVENTTARGET": "",  # from DevTools
        "__EVENTARGUMENT": "",
        "__LASTFOCUS": "",
        # "ctl00$ContentPlaceHolder1$PressRlsList1$CustomPager1$dlCustomPager$ctl05$lnkbtnPaging" : 6,
        "ctl00$ContentPlaceHolder1$PressRlsList1$CustomPager1$ibtnMoveNext.x": 4,
        "ctl00$ContentPlaceHolder1$PressRlsList1$CustomPager1$ibtnMoveNext.y" : 18
    }

    resp2 = session.post(url, data=payload)
    soup = BeautifulSoup(resp2.text, "html.parser")
    # print(soup2)

    for row in soup.find('tbody').find_all('tr'):
        s_no = row.find_all('td')[0].text.strip()
        newsDate = row.find_all('td')[1].text.strip()
        unit = row.find_all('td')[2].text.strip()
        category = row.find_all('td')[3].text.strip()
        heading = row.find_all('td')[4].text.strip()
        link = "https://nia.gov.in" +row.find_all('td')[5].find('a')['href']
        # print({"s_no": s_no, "heading": heading, "newsDate": newsDate, "link" : link})
        newsData = {"heading": heading, "newsDate": newsDate, "link" : link}
        print(newsData)
        write_to_txt_resolve(newsData)
        print("-----------------------------------------")
        



{'heading': '3 Maoist Operatives Chargesheeted by NIA in Chhattisgarh Terror Funding Case', 'newsDate': '25/08/2025', 'link': 'https://nia.gov.in/writereaddata/Portal/PressReleaseNew/2004_1_PR25082025.pdf'}
-----------------------------------------
{'heading': 'NIA Searches 9 Locations in TN & Arrests 1 Harbourer of POs In 2019 Ramalingam Murder Case', 'newsDate': '21/08/2025', 'link': 'https://nia.gov.in/writereaddata/Portal/PressReleaseNew/2003_1_PR21082025.pdf'}
-----------------------------------------
{'heading': 'Another Accused Sentenced to RI by NIA Spl Court in Mumbra Fake Currency Case', 'newsDate': '18/08/2025', 'link': 'https://nia.gov.in/writereaddata/Portal/PressReleaseNew/2002_1_PR18082025.pdf'}
-----------------------------------------
{'heading': 'Key Accused in 2014 Sakir Hussain Espionage Sentenced to 5 Yrs RI by NIA Spl Court', 'newsDate': '14/08/2025', 'link': 'https://nia.gov.in/writereaddata/Portal/PressReleaseNew/1998_1_PR140820254.pdf'}
------------------------

In [1]:
##### Code to filter only cin and company name present data ####################
import re
import json

# === Input / Output paths ===
input_file = r"D:\Nexensus_Projects\News\data_CBI_mapped.txt"
output_file = r"D:\Nexensus_Projects\News\data_CBI_mapped_cleaned.txt"

# === Read file ===
with open(input_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

# === Prepare storage ===
cleaned_records = []

# === Process each JSON line ===
for line in lines:
    line = line.strip()
    if not line:
        continue

    # Remove trailing comma if present
    json_str = re.sub(r',\s*$', '', line)

    try:
        data = json.loads(json_str)
    except json.JSONDecodeError:
        continue  # skip malformed JSON

    # Skip if companyName is null, empty, or missing
    if not data.get("companyName"):
        continue

    cleaned_records.append(data)

# === Write cleaned data to new file ===
with open(output_file, "w", encoding="utf-8") as f:
    for record in cleaned_records:
        json.dump(record, f, ensure_ascii=False)
        f.write(",\n")

print(f"✅ Cleaned file saved successfully: {output_file}")
print(f"Total valid records: {len(cleaned_records)}")


✅ Cleaned file saved successfully: D:\Nexensus_Projects\News\data_CBI_mapped_cleaned.txt
Total valid records: 42
